# Optional Project - Colab muP Attention Scale Ablation

Runs the ablation for Part 3: keep the muP optimizer/base-shape/readout setup fixed, and compare tiny/small models with attention scale `1/head_dim` versus the standard PyTorch attention scale `1/sqrt(head_dim)`. Re-run interrupted training cells to resume from Google Drive checkpoints.

In [ ]:
# ===== User config =====
REPO_URL = "https://github.com/Peng-y-x/optionalproject.git"
REPO_DIR = "/content/optionalproject"
REPO_BRANCH = "run"

ABLATION_CONFIG = "configs/mup_attention_scale_ablation.yaml"
MUP_BEST_LR_JSON = "outputs/part3_mup_lr_sweep/best_lr.json"
DRIVE_MUP_BEST_LR_JSON = "/content/drive/MyDrive/svg-scaling/part3_mup_lr_sweep/best_lr.json"

DRIVE_ROOT = "/content/drive/MyDrive/svg-scaling"
ABLATION_RUN_DIR = "part3_attention_scale_ablation"
ABLATION_ANALYSIS_DIR = "part3_attention_scale_ablation_analysis"

# Set to None to use the LR from MUP_BEST_LR_JSON if available, otherwise config default.
LEARNING_RATE_OVERRIDE = None


In [ ]:
# 1) Clone repo and checkout branch
import os
if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)
%cd $REPO_DIR
!git fetch origin
!git checkout {REPO_BRANCH}
!git pull origin {REPO_BRANCH}
!git branch --show-current
!git rev-parse --short HEAD


In [ ]:
# 2) Mount Google Drive for resumable ablation checkpoints and outputs
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {DRIVE_ROOT}/{ABLATION_RUN_DIR}
!mkdir -p {DRIVE_ROOT}/{ABLATION_ANALYSIS_DIR}


In [ ]:
# 3) Install system + Python dependencies
!apt-get update -y
!apt-get install -y libcairo2 libcairo2-dev libffi-dev
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
# 4) HF auth from Colab Keys (key name must be HF_TOKEN)
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    print('HF token loaded from Colab key.')
else:
    print('HF token not found in Colab key HF_TOKEN. Public dataset loading may still work.')
print('has_hf_token:', bool(os.getenv('HF_TOKEN')))


In [ ]:
# 5) GPU sanity check
import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('bf16_supported', torch.cuda.is_bf16_supported())


In [ ]:
# 6) Reuse the Part 3 selected muP LR if it exists on Drive
import json, shutil
from pathlib import Path

if not Path(MUP_BEST_LR_JSON).exists() and Path(DRIVE_MUP_BEST_LR_JSON).exists():
    Path(MUP_BEST_LR_JSON).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_MUP_BEST_LR_JSON, MUP_BEST_LR_JSON)

BEST_LR_ARG = ''
LR_ARG = ''
if LEARNING_RATE_OVERRIDE is not None:
    LR_ARG = f'--learning-rate {LEARNING_RATE_OVERRIDE}'
elif Path(MUP_BEST_LR_JSON).exists():
    best = json.loads(Path(MUP_BEST_LR_JSON).read_text())
    print(json.dumps(best, indent=2))
    BEST_LR_ARG = f'--best-lr-json {MUP_BEST_LR_JSON}'
else:
    print('No Part 3 best LR JSON found; using the ablation config default learning_rate.')

print('BEST_LR_ARG=', BEST_LR_ARG)
print('LR_ARG=', LR_ARG)


## Optional Smoke Test

For a quick wiring check, temporarily uncomment `max_train_tokens` and `max_eval_tokens` in `configs/mup_attention_scale_ablation.yaml`. For final results, leave them commented so each run uses the same one-epoch protocol as Part 3.

In [ ]:
# 7) Run tiny/small ablation
# If Colab disconnects, rerun this cell; each run resumes from Drive latest.pt.
%cd $REPO_DIR
!python scripts/run_mup_attention_scale_ablation.py \
  --config {ABLATION_CONFIG} \
  {BEST_LR_ARG} \
  {LR_ARG}


In [ ]:
# 8) Inspect ablation results
import pandas as pd
from pathlib import Path
from IPython.display import display, Image

analysis_dir = Path('outputs/part3_attention_scale_ablation_analysis')
csv_path = analysis_dir / 'attention_scale_ablation.csv'
png_path = analysis_dir / 'attention_scale_ablation.png'

df = pd.read_csv(csv_path)
display(df[['source_config', 'attention_scale', 'learning_rate', 'num_parameters', 'val_loss', 'val_ppl', 'tokens_seen']])
display(Image(filename=str(png_path)))
